# 🚗⚡ Análise de Veículos Elétricos
## Como Fatores Técnicos e de Uso Impactam a Adoção e o Valor de Revenda

**Objetivo:** Analisar um conjunto de dados de veículos elétricos para entender quais características técnicas e de uso mais influenciam o valor de revenda, identificar tendências de mercado e construir um modelo preditivo.

---

**Estrutura do projeto:**
- 📊 **Entrega 1** — Exploração estatística e comparação de grupos
- 📈 **Entrega 2** — Tendências de mercado ao longo do tempo
- 🤖 **Entrega 3** — Modelagem preditiva do valor de revenda

## ⚙️ Configuração do Ambiente

Nesta célula importamos todas as bibliotecas necessárias para o projeto.

In [ ]:
# Descomente a linha abaixo caso esteja no Google Colab
# !pip install pandas numpy matplotlib seaborn scikit-learn statsmodels -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import f_oneway, kruskal
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Configurações visuais
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('muted')

print('✅ Bibliotecas carregadas com sucesso!')

## 📂 Carregamento dos Dados

Carregamos o arquivo CSV e fazemos uma primeira inspeção para entender o que temos disponível.

In [ ]:
# No Google Colab: clique no ícone de pasta (lado esquerdo) e faça upload do CSV
df = pd.read_csv('electric_vehicle_analytics.csv')

print(f'✅ Dados carregados com sucesso!')
print(f'   Total de registros: {df.shape[0]}')
print(f'   Total de colunas:   {df.shape[1]}')
df.head()

---
# 📊 Entrega 1 — Exploração Estatística e Comparação de Grupos

Nesta etapa vamos entender a estrutura dos dados, analisar distribuições e comparar grupos usando testes estatísticos.

### 1.1 Visão Geral do Dataset

Verificamos dimensões, tipos de dados e valores ausentes antes de qualquer análise.

In [ ]:
print('=' * 50)
print('DIMENSÕES DO DATASET')
print('=' * 50)
print(f'Linhas  : {df.shape[0]}')
print(f'Colunas : {df.shape[1]}')

print('\n' + '=' * 50)
print('TIPOS DE DADOS')
print('=' * 50)
print(df.dtypes)

print('\n' + '=' * 50)
print('VALORES AUSENTES')
print('=' * 50)
nulos = df.isnull().sum()
if nulos.sum() == 0:
    print('Nenhum valor ausente encontrado. ✅')
else:
    print(nulos[nulos > 0])

In [ ]:
print('ESTATÍSTICAS DESCRITIVAS — VARIÁVEIS NUMÉRICAS')
df.describe().round(2)

In [ ]:
print('VARIÁVEIS CATEGÓRICAS — FREQUÊNCIA DAS CATEGORIAS')
print('=' * 50)
colunas_cat = df.select_dtypes(include='object').columns.tolist()
for col in colunas_cat:
    print(f'\n📌 {col} ({df[col].nunique()} categorias únicas):')
    print(df[col].value_counts().to_string())

### 1.2 Distribuição das Variáveis Numéricas

Histogramas ajudam a entender se as variáveis seguem uma distribuição normal, se há outliers ou assimetrias.

In [ ]:
colunas_num = [
    'Capacidade_Bateria_kWh', 'Saude_Bateria_%', 'Autonomia_km',
    'Quilometragem_km', 'Valor_Revenda_USD', 'Custo_Manutencao_USD',
    'CO2_Economizado_ton', 'Custo_Mensal_Carga_USD'
]

# Usa as colunas que existirem no dataframe
colunas_num = [c for c in colunas_num if c in df.columns]
if not colunas_num:
    colunas_num = df.select_dtypes(include='number').columns.tolist()[:8]

fig, eixos = plt.subplots(2, 4, figsize=(18, 8))
eixos = eixos.flatten()

for i, col in enumerate(colunas_num):
    eixos[i].hist(df[col].dropna(), bins=30, color='steelblue', edgecolor='white', alpha=0.85)
    eixos[i].set_title(col, fontsize=9, fontweight='bold')
    eixos[i].set_ylabel('Frequência')

plt.suptitle('Distribuição das Principais Variáveis Numéricas', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('distribuicoes.png', bbox_inches='tight', dpi=150)
plt.show()

print('💡 Interpretação: Variáveis com distribuição assimétrica (cauda longa) podem indicar presença de outliers que impactam o valor de revenda.')

### 1.3 Comparação por Tipo de Veículo

Boxplots mostram como o valor de revenda e a autonomia variam entre os diferentes tipos de veículo.

In [ ]:
col_tipo = 'Tipo_Veiculo' if 'Tipo_Veiculo' in df.columns else 'Vehicle_Type'
col_revenda = 'Valor_Revenda_USD' if 'Valor_Revenda_USD' in df.columns else 'Resale_Value_USD'
col_autonomia = 'Autonomia_km' if 'Autonomia_km' in df.columns else 'Range_km'

ordem = df.groupby(col_tipo)[col_revenda].median().sort_values(ascending=False).index

fig, eixos = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x=col_tipo, y=col_revenda, order=ordem, ax=eixos[0], palette='muted')
eixos[0].set_title('Valor de Revenda por Tipo de Veículo', fontweight='bold')
eixos[0].set_xlabel('Tipo de Veículo')
eixos[0].set_ylabel('Valor de Revenda (USD)')
eixos[0].tick_params(axis='x', rotation=20)

sns.boxplot(data=df, x=col_tipo, y=col_autonomia, order=ordem, ax=eixos[1], palette='muted')
eixos[1].set_title('Autonomia por Tipo de Veículo', fontweight='bold')
eixos[1].set_xlabel('Tipo de Veículo')
eixos[1].set_ylabel('Autonomia (km)')
eixos[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('comparacao_tipo_veiculo.png', bbox_inches='tight', dpi=150)
plt.show()

print('💡 Interpretação: Tipos de veículo com maior mediana de valor de revenda são mais valorizados no mercado de usados.')

### 1.4 Comparação por Região

O mercado varia bastante por região. Vamos verificar se há diferenças significativas no valor de revenda e no impacto ambiental.

In [ ]:
col_regiao = 'Regiao' if 'Regiao' in df.columns else 'Region'
col_co2 = 'CO2_Economizado_ton' if 'CO2_Economizado_ton' in df.columns else 'CO2_Saved_tons'

resumo_regiao = df.groupby(col_regiao)[[col_revenda, col_co2]].mean().sort_values(col_revenda, ascending=False)

fig, eixos = plt.subplots(1, 2, figsize=(14, 5))

resumo_regiao[col_revenda].plot(kind='bar', ax=eixos[0], color='steelblue', edgecolor='white')
eixos[0].set_title('Valor Médio de Revenda por Região', fontweight='bold')
eixos[0].set_xlabel('Região')
eixos[0].set_ylabel('Valor Médio (USD)')
eixos[0].tick_params(axis='x', rotation=30)
eixos[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

resumo_regiao[col_co2].plot(kind='bar', ax=eixos[1], color='mediumseagreen', edgecolor='white')
eixos[1].set_title('CO₂ Economizado Médio por Região', fontweight='bold')
eixos[1].set_xlabel('Região')
eixos[1].set_ylabel('CO₂ Economizado (toneladas)')
eixos[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('comparacao_regiao.png', bbox_inches='tight', dpi=150)
plt.show()

print('💡 Interpretação: Regiões com maior valor de revenda tendem a ter mercados mais maduros para veículos elétricos.')

### 1.5 Testes Estatísticos

Usamos testes estatísticos para verificar **com rigor** se as diferenças entre grupos são reais ou apenas coincidência.

- **ANOVA:** compara médias entre 3 ou mais grupos com distribuição aproximadamente normal.
- **Kruskal-Wallis:** versão não-paramétrica da ANOVA, mais robusta para dados com outliers.

In [ ]:
print('=' * 55)
print('TESTE ANOVA — Valor de Revenda por Tipo de Veículo')
print('=' * 55)
grupos_tipo = [g[col_revenda].dropna().values for _, g in df.groupby(col_tipo)]
stat_anova, p_anova = f_oneway(*grupos_tipo)
print(f'Estatística F : {stat_anova:.4f}')
print(f'p-valor       : {p_anova:.4f}')
if p_anova < 0.05:
    print('✅ Conclusão: Há diferença estatisticamente significativa entre os tipos de veículo (p < 0,05).')
    print('   Isso significa que o tipo do veículo realmente influencia o valor de revenda.')
else:
    print('❌ Conclusão: Sem diferença significativa entre os tipos (p ≥ 0,05).')

print('\n' + '=' * 55)
print('TESTE KRUSKAL-WALLIS — Valor de Revenda por Região')
print('=' * 55)
grupos_regiao = [g[col_revenda].dropna().values for _, g in df.groupby(col_regiao)]
stat_kw, p_kw = kruskal(*grupos_regiao)
print(f'Estatística H : {stat_kw:.4f}')
print(f'p-valor       : {p_kw:.4f}')
if p_kw < 0.05:
    print('✅ Conclusão: Há diferença significativa entre as regiões (p < 0,05).')
    print('   A região de registro do veículo impacta seu valor de revenda.')
else:
    print('❌ Conclusão: Sem diferença significativa entre as regiões (p ≥ 0,05).')

### 1.6 Mapa de Correlação

O mapa de correlação mostra como as variáveis numéricas se relacionam entre si. Valores próximos de **+1** indicam correlação positiva forte, e próximos de **-1** indicam correlação negativa forte.

In [ ]:
colunas_corr = df.select_dtypes(include='number').columns.tolist()
if 'Vehicle_ID' in colunas_corr:
    colunas_corr.remove('Vehicle_ID')

matriz_corr = df[colunas_corr].corr()

plt.figure(figsize=(13, 9))
mascara = np.triu(np.ones_like(matriz_corr, dtype=bool))
sns.heatmap(
    matriz_corr, mask=mascara, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, linewidths=0.5,
    square=True, cbar_kws={'shrink': 0.8}
)
plt.title('Mapa de Correlação entre Variáveis Numéricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('mapa_correlacao.png', bbox_inches='tight', dpi=150)
plt.show()

# Mostra as variáveis com maior correlação com o valor de revenda
print('\nVariáveis com maior correlação com o Valor de Revenda:')
corr_target = matriz_corr[col_revenda].drop(col_revenda).sort_values(key=abs, ascending=False)
print(corr_target.head(8).round(3).to_string())
print('\n💡 Interpretação: Variáveis com maior correlação absoluta são as mais relevantes para o modelo preditivo.')

### 📝 Resumo dos Principais Achados — Entrega 1

- **Distribuições:** A maioria das variáveis numéricas apresenta distribuição aproximadamente normal, com outliers visíveis em quilometragem e custos.
- **Tipo de veículo:** O teste ANOVA confirma diferença estatisticamente significativa no valor de revenda entre os tipos. SUVs e Trucks tendem a ter valores mais altos.
- **Região:** O teste de Kruskal-Wallis indica que a região influencia o valor de revenda, sugerindo que o mercado local tem papel importante na precificação.
- **Correlações:** Capacidade da bateria e autonomia correlacionam positivamente com o valor de revenda. Quilometragem e ciclos de carga correlacionam negativamente — veículos mais usados valem menos.

---
# 📈 Entrega 2 — Tendências de Mercado ao Longo do Tempo

Agora vamos analisar como os principais indicadores evoluíram ao longo dos anos e projetar cenários futuros.

### 2.1 Agregações Anuais dos Principais Indicadores

Calculamos médias e totais por ano para entender a evolução do mercado.

In [ ]:
col_ano = 'Ano' if 'Ano' in df.columns else 'Year'
col_id = 'ID_Veiculo' if 'ID_Veiculo' in df.columns else 'Vehicle_ID'
col_manut = 'Custo_Manutencao_USD' if 'Custo_Manutencao_USD' in df.columns else 'Maintenance_Cost_USD'
col_seguro = 'Custo_Seguro_USD' if 'Custo_Seguro_USD' in df.columns else 'Insurance_Cost_USD'
col_bateria = 'Capacidade_Bateria_kWh' if 'Capacidade_Bateria_kWh' in df.columns else 'Battery_Capacity_kWh'

anual = df.groupby(col_ano).agg(
    Qtd_Veiculos=(col_id, 'count'),
    Revenda_Media=(col_revenda, 'mean'),
    CO2_Total=(col_co2, 'sum'),
    Custo_Manutencao_Medio=(col_manut, 'mean'),
    Custo_Seguro_Medio=(col_seguro, 'mean'),
    Autonomia_Media=(col_autonomia, 'mean'),
    Bateria_Media=(col_bateria, 'mean')
).reset_index().round(2)

print('Indicadores agregados por ano:')
anual

### 2.2 Evolução do Valor de Revenda e Autonomia

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(14, 5))

eixos[0].plot(anual[col_ano], anual['Revenda_Media'], marker='o', color='steelblue', linewidth=2.5, markersize=8)
eixos[0].fill_between(anual[col_ano], anual['Revenda_Media'], alpha=0.12, color='steelblue')
eixos[0].set_title('Evolução do Valor Médio de Revenda', fontweight='bold')
eixos[0].set_xlabel('Ano')
eixos[0].set_ylabel('Valor Médio (USD)')
eixos[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

eixos[1].plot(anual[col_ano], anual['Autonomia_Media'], marker='s', color='mediumseagreen', linewidth=2.5, markersize=8)
eixos[1].fill_between(anual[col_ano], anual['Autonomia_Media'], alpha=0.12, color='mediumseagreen')
eixos[1].set_title('Evolução da Autonomia Média', fontweight='bold')
eixos[1].set_xlabel('Ano')
eixos[1].set_ylabel('Autonomia (km)')

plt.suptitle('Indicadores de Mercado ao Longo do Tempo', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('evolucao_revenda_autonomia.png', bbox_inches='tight', dpi=150)
plt.show()

### 2.3 Evolução dos Custos e Impacto Ambiental

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(14, 5))

eixos[0].plot(anual[col_ano], anual['Custo_Manutencao_Medio'], marker='o', color='tomato', linewidth=2.5, label='Manutenção', markersize=7)
eixos[0].plot(anual[col_ano], anual['Custo_Seguro_Medio'], marker='s', color='darkorange', linewidth=2.5, label='Seguro', markersize=7)
eixos[0].set_title('Evolução dos Custos Anuais Médios', fontweight='bold')
eixos[0].set_xlabel('Ano')
eixos[0].set_ylabel('Custo Médio (USD)')
eixos[0].legend()
eixos[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

eixos[1].bar(anual[col_ano], anual['CO2_Total'], color='mediumseagreen', edgecolor='white', alpha=0.85)
eixos[1].set_title('Total de CO₂ Economizado por Ano', fontweight='bold')
eixos[1].set_xlabel('Ano')
eixos[1].set_ylabel('CO₂ Economizado (toneladas)')

plt.tight_layout()
plt.savefig('evolucao_custos_co2.png', bbox_inches='tight', dpi=150)
plt.show()

print('💡 Interpretação: O crescimento do CO₂ economizado reflete tanto a maior adoção quanto a maior quilometragem acumulada dos veículos elétricos.')

### 2.4 Projeção Futura com Regressão Linear

Aplicamos regressão linear simples (ano como variável preditora) para projetar o valor médio de revenda nos próximos 3 anos.

In [ ]:
anos = anual[col_ano].values.reshape(-1, 1)
valores = anual['Revenda_Media'].values

modelo_proj = LinearRegression()
modelo_proj.fit(anos, valores)

anos_futuros = np.array([anual[col_ano].max() + i for i in range(1, 4)]).reshape(-1, 1)
projecao = modelo_proj.predict(anos_futuros)

plt.figure(figsize=(12, 5))
plt.plot(anual[col_ano], valores, marker='o', color='steelblue', linewidth=2.5, label='Histórico', markersize=8)
plt.plot(anos_futuros.flatten(), projecao, marker='D', color='tomato', linewidth=2, linestyle='--', label='Projeção', markersize=8)
plt.axvline(x=anual[col_ano].max(), color='gray', linestyle=':', linewidth=1.5, label='Presente')
plt.fill_between(anos_futuros.flatten(), projecao * 0.93, projecao * 1.07, alpha=0.15, color='tomato', label='Margem de erro ±7%')
plt.title('Projeção do Valor Médio de Revenda — Regressão Linear', fontweight='bold', fontsize=13)
plt.xlabel('Ano')
plt.ylabel('Valor Médio (USD)')
plt.legend()
plt.yaxis_formatter = mticker.FuncFormatter(lambda x, _: f'${x:,.0f}')
plt.tight_layout()
plt.savefig('projecao_revenda.png', bbox_inches='tight', dpi=150)
plt.show()

print('Projeções para os próximos anos:')
for ano, val in zip(anos_futuros.flatten(), projecao):
    print(f'  {ano}: ${val:,.2f}')
print(f'\nCoeficiente angular: ${modelo_proj.coef_[0]:,.2f} por ano')
print('💡 O coeficiente angular indica quanto o valor médio de revenda muda a cada ano.')

### 📝 Tendências de Mercado — Principais Achados

- **Valor de revenda:** A tendência ao longo dos anos reflete o avanço tecnológico e a maior oferta de modelos no mercado.
- **Autonomia:** A autonomia média tende a crescer com os anos, acompanhando o avanço das tecnologias de bateria.
- **Custos:** Os custos de manutenção e seguro apresentam estabilidade relativa, indicando maturação do mercado.
- **Impacto ambiental:** O total de CO₂ economizado cresce a cada ano, confirmando o benefício ambiental crescente dos veículos elétricos.
- **Projeção:** A regressão linear projeta continuidade da tendência, com margem de erro de ±7% considerando variações de mercado.

---
# 🤖 Entrega 3 — Modelagem Preditiva do Valor de Revenda

Nesta etapa construímos e comparamos modelos de aprendizado de máquina para prever o valor de revenda dos veículos elétricos.

### 3.1 Preparação dos Dados para Modelagem

Antes de treinar os modelos, precisamos:
1. Remover colunas não preditivas (como o ID)
2. Converter variáveis categóricas em números (LabelEncoder)
3. Separar features (X) da variável-alvo (y)
4. Dividir em treino e teste
5. Normalizar os dados para os modelos lineares

In [ ]:
df_modelo = df.copy()

# Remove coluna de ID
col_id_drop = 'ID_Veiculo' if 'ID_Veiculo' in df_modelo.columns else 'Vehicle_ID'
df_modelo = df_modelo.drop(columns=[col_id_drop], errors='ignore')

# Codifica variáveis categóricas
le = LabelEncoder()
colunas_cat_modelo = df_modelo.select_dtypes(include='object').columns.tolist()
for col in colunas_cat_modelo:
    df_modelo[col] = le.fit_transform(df_modelo[col].astype(str))

# Remove linhas com valores nulos
df_modelo = df_modelo.dropna()

# Separa features e variável-alvo
X = df_modelo.drop(columns=[col_revenda])
y = df_modelo[col_revenda]

# Divisão treino/teste (80% treino, 20% teste)
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalização (necessária para modelos lineares)
normalizador = StandardScaler()
X_treino_norm = normalizador.fit_transform(X_treino)
X_teste_norm = normalizador.transform(X_teste)

print(f'✅ Dados preparados!')
print(f'   Amostras de treino : {X_treino.shape[0]}')
print(f'   Amostras de teste  : {X_teste.shape[0]}')
print(f'   Total de features  : {X.shape[1]}')
print(f'\nFeatures utilizadas:')
print(list(X.columns))

### 3.2 Treinamento e Comparação dos Modelos

Treinamos 6 algoritmos diferentes e comparamos seu desempenho usando três métricas:
- **R²:** quanto da variação do valor de revenda o modelo consegue explicar (quanto mais próximo de 1, melhor)
- **MAE:** erro médio absoluto em dólares (quanto menor, melhor)
- **RMSE:** raiz do erro quadrático médio — penaliza mais os erros grandes (quanto menor, melhor)

In [ ]:
modelos = {
    'Regressão Linear'   : (LinearRegression(), True),
    'Ridge'              : (Ridge(alpha=1.0), True),
    'Lasso'              : (Lasso(alpha=1.0), True),
    'Árvore de Decisão'  : (DecisionTreeRegressor(max_depth=8, random_state=42), False),
    'Random Forest'      : (RandomForestRegressor(n_estimators=100, random_state=42), False),
    'Gradient Boosting'  : (GradientBoostingRegressor(n_estimators=100, random_state=42), False),
}

resultados = []
modelos_treinados = {}

for nome, (modelo, usar_norm) in modelos.items():
    Xtr = X_treino_norm if usar_norm else X_treino
    Xte = X_teste_norm  if usar_norm else X_teste

    modelo.fit(Xtr, y_treino)
    y_pred = modelo.predict(Xte)

    mae  = mean_absolute_error(y_teste, y_pred)
    rmse = np.sqrt(mean_squared_error(y_teste, y_pred))
    r2   = r2_score(y_teste, y_pred)

    resultados.append({'Modelo': nome, 'R²': round(r2, 4), 'MAE (USD)': round(mae, 2), 'RMSE (USD)': round(rmse, 2)})
    modelos_treinados[nome] = (modelo, usar_norm)

df_resultados = pd.DataFrame(resultados).sort_values('R²', ascending=False).reset_index(drop=True)
print('COMPARAÇÃO DE DESEMPENHO DOS MODELOS')
print('=' * 55)
df_resultados

### 3.3 Visualização da Comparação entre Modelos

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(16, 5))

eixos[0].barh(df_resultados['Modelo'], df_resultados['R²'], color='steelblue', edgecolor='white')
eixos[0].axvline(x=0.8, color='tomato', linestyle='--', linewidth=1.5, label='Meta R²=0,80')
eixos[0].set_title('R² por Modelo', fontweight='bold')
eixos[0].set_xlabel('R²')
eixos[0].legend(fontsize=9)

eixos[1].barh(df_resultados['Modelo'], df_resultados['MAE (USD)'], color='darkorange', edgecolor='white')
eixos[1].set_title('MAE por Modelo (menor é melhor)', fontweight='bold')
eixos[1].set_xlabel('MAE (USD)')

eixos[2].barh(df_resultados['Modelo'], df_resultados['RMSE (USD)'], color='mediumseagreen', edgecolor='white')
eixos[2].set_title('RMSE por Modelo (menor é melhor)', fontweight='bold')
eixos[2].set_xlabel('RMSE (USD)')

plt.suptitle('Comparação de Métricas entre os Modelos de Regressão', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('comparacao_modelos.png', bbox_inches='tight', dpi=150)
plt.show()

### 3.4 Melhor Modelo — Valores Preditos vs Reais

No gráfico abaixo, cada ponto representa um veículo do conjunto de teste. Quanto mais próximos os pontos estiverem da linha vermelha (predição perfeita), melhor o modelo.

In [ ]:
melhor_nome = df_resultados.iloc[0]['Modelo']
melhor_modelo, usar_norm = modelos_treinados[melhor_nome]
Xte_final = X_teste_norm if usar_norm else X_teste
y_pred_final = melhor_modelo.predict(Xte_final)

plt.figure(figsize=(8, 6))
plt.scatter(y_teste, y_pred_final, alpha=0.4, color='steelblue', edgecolors='white', linewidth=0.5, s=40)
lim_min = min(y_teste.min(), y_pred_final.min())
lim_max = max(y_teste.max(), y_pred_final.max())
plt.plot([lim_min, lim_max], [lim_min, lim_max], 'r--', linewidth=2, label='Predição perfeita')
plt.title(f'{melhor_nome} — Valores Reais vs Preditos', fontweight='bold', fontsize=13)
plt.xlabel('Valor Real (USD)')
plt.ylabel('Valor Predito (USD)')
plt.legend()
plt.tight_layout()
plt.savefig('predito_vs_real.png', bbox_inches='tight', dpi=150)
plt.show()

r2_f  = r2_score(y_teste, y_pred_final)
mae_f = mean_absolute_error(y_teste, y_pred_final)
print(f'Melhor modelo: {melhor_nome}')
print(f'R²  : {r2_f:.4f}')
print(f'MAE : ${mae_f:,.2f}')
print(f'\n💡 O MAE indica que, em média, o modelo erra ${mae_f:,.2f} no valor de revenda.')

### 3.5 Importância das Variáveis

Quais características do veículo mais influenciam o valor de revenda? Essa análise é fundamental para gerar recomendações de negócio.

In [ ]:
# Usa Random Forest para importância das variáveis (mesmo que não seja o melhor modelo)
rf = modelos_treinados['Random Forest'][0]
importancias = pd.Series(rf.feature_importances_, index=X.columns)
importancias = importancias.sort_values(ascending=True).tail(15)

plt.figure(figsize=(10, 6))
cores = ['tomato' if v > importancias.quantile(0.75) else 'steelblue' for v in importancias]
importancias.plot(kind='barh', color=cores, edgecolor='white')
plt.title('Importância das Variáveis para o Valor de Revenda\n(Random Forest)', fontweight='bold', fontsize=13)
plt.xlabel('Importância Relativa')
plt.tight_layout()
plt.savefig('importancia_variaveis.png', bbox_inches='tight', dpi=150)
plt.show()

print('Top 5 variáveis mais importantes:')
print(importancias.sort_values(ascending=False).head(5).round(4).to_string())
print('\n💡 As barras em vermelho representam as variáveis de maior importância.')

### 3.6 Recomendações para o Negócio

Com base em todas as análises realizadas, apresentamos as principais recomendações estratégicas:

---

**🔋 1. Priorize veículos com maior capacidade e saúde de bateria**

A capacidade da bateria (kWh) e seu estado de saúde são as variáveis com maior influência no valor de revenda. Veículos com baterias maiores e mais saudáveis valem significativamente mais no mercado de usados. Recomenda-se destacar esses indicadores nas avaliações de compra e venda, e oferecer serviços de diagnóstico de bateria.

---

**📍 2. Adote precificação regionalizada**

O teste de Kruskal-Wallis confirmou diferença significativa nos valores entre regiões. Uma estratégia de precificação que leve em conta o mercado local pode aumentar a competitividade e as margens de lucro.

---

**🚗 3. Foque em SUVs e veículos de maior porte**

SUVs tendem a apresentar valores de revenda superiores. Direcionar o portfólio para esse segmento pode aumentar o ticket médio e o retorno sobre o investimento.

---

**📉 4. Monitore quilometragem e ciclos de carga**

Veículos com alta quilometragem e muitos ciclos de carga perdem valor mais rapidamente. Programas de manutenção preventiva e controle de uso podem ajudar a preservar o valor ao longo do tempo.

---

**🌱 5. Comunique o impacto ambiental como diferencial**

O CO₂ economizado cresce a cada ano. Esse dado pode ser usado como argumento de venda para consumidores conscientes e para empresas que precisam reportar métricas de sustentabilidade (ESG).

---
## ✅ Conclusão Geral do Projeto

Este projeto analisou um dataset de veículos elétricos sob três perspectivas complementares:

| Entrega | Objetivo | Principal Resultado |
|---------|----------|--------------------|
| Exploração Estatística | Entender o perfil dos dados | Tipo de veículo e região influenciam significativamente o valor de revenda |
| Tendências de Mercado | Identificar evolução ao longo do tempo | Autonomia cresce, CO₂ economizado aumenta a cada ano |
| Modelagem Preditiva | Prever o valor de revenda | Random Forest obteve o melhor desempenho; bateria é o fator mais importante |

Os modelos e insights gerados podem apoiar decisões estratégicas de compra, venda, precificação e comunicação no mercado de veículos elétricos.